# Práctica 2
#### **Grupo Q**
Marc Martínez Arias, Pedro Barros Bobadilla

[URL del repositorio](https://github.com/Code-Cram/PROGRAMACION_CONCURRENTE_Y_DISTRIBUIDA_Grupo_Q)

Vamos a desarrollar este juego de tipo Multi User Dungeon (MUD)teniendo varias cosas en cuenta:

#### Primero:
El juego está basado en texto. Aun así, vamos a modelar un mapa de rol para desarrollar el juego en él. Este mapa ha de estar ambientado en la edad media, disponiendo de localizaciones que se unen entre sí donde se desarrolla un comercio.

#### Segundo:
Cada miembro que participe en el juego ostenta una profesión la cual gestiona y coordina sus pedidos, procesos, tiempos, precios y productos. En cada calle donde habrá 2 estudiantes que ostentan una profesión. Estas profesiones han de estar relacionadas entre sí. Por ejemplo, habrá una calle de herrería, donde haya un negocio de armaduras y una forja. Toda profesión tiene sus tareas independientes. Es decir, si el que se dedica a las armaduras quiere vender una cota de malla necesita hierro procesado del herrero. 

**IMPORTANTE!!!**

El código de una misma calle deberá poderse ejecutar en una sola máquina, usando, por ejemplo, colas para la comunicación y el paso de recursos.

#### Tercero:
Existen barrios formados por dos calles. Es decir, 4 estudiantes, 4 productores. Y hay dos tipos de barrios:

- El barrio comercial deberá decidir cómo distribuir los recursos generados por campesinos y otros comercios y almacenar los recursos de otros pueblos. El grupo deberá implementar un mercado o similar para ello. Se anima a buscar soluciones creativas, por ejemplo, un sistema de trueque.

- El barrio económico deberá decidir cómo gestionar la contabilidad monetaria o económica de cada pueblo. El barrio deberá implementar un sistema contable para cada pueblo. Se anima a buscar soluciones creativas, por ejemplo, una renta básica o monedas sociales locales.

**IMPORTANTE!!!**

El código de distintas calles se ejecuta en distintos ordenadores, por lo que se deberán usar herramientas de la programación distribuida, como las invocaciones remotas para implementar la solución.

#### Cuarto:
Los pueblos están compuestos por dos barrios, uno de cada tipo. Es decir, 4 calles, 8 estudiantes. Al tener los dos tipos de barrio cada pueblo se encarga de como gestionar su economía y sus recursos. Por ejemplo, incluyendo mejoras fuera de los barrios como puede ser un sistema de gestión de la oferta y la demanda. Es decir, si tenemos una calle de panaderos es tarea del barrio económico poner un precio fijo de 1 moneda por ejemplo. En el momento en el que se supere cierto número de panes almacenados, ya sea 50 panes es tarea del barrio económico bajar el precio del pan debido a la gran oferta del pan del pueblo y viceversa.

#### **Se tiene en cuenta lo siguiente para la práctica:**
El proceso de la creación de este juego está preparado para que participen hasta 8 personas con sus distintos turnos y opciones. Para la inclusión de cada participante en dicho juego se tiene cuenta a la hora de definir las distintas clases que los trabajadores definan su rol y sus métodos, donde se incluye la gestión de pedidos, procesos, tiempos, precios y productos. Luego se incluyen estos trabajadores en sus calles que estarán preparadas para incluir dos estudiantes con profesiones en común y así sucesivamente hasta llegar a la creación de pueblo. Para utilizar la concurrencia cada trabajador funcionará como un hilo. 

Vamos a poner un ejemplo de cómo se estructurará la ejecución para respetar la naturaleza distribuida e interactiva del MUD. En lugar de iniciar un único programa que pregunte por todos los jugadores, cada máquina ejecutará su propio script representando, como mínimo, una calle con sus 2 estudiantes. Una vez decidido el tipo de juego, en cada script (nodo) la concurrencia se dividirá en dos planos:

- **Plano de producción:** Cada profesión será un hilo independiente que funcionará de forma semiautomática. Por ejemplo, el herrero comprobará si tiene carbón, hará un `time.sleep()` simulando su tiempo de creación, y dejará el hierro procesado en un inventario protegido por un `Lock` o una `queue.Queue` dependiendo de la elección de la memoria a implementar.

- **Plano de interacción:** El hilo principal de la terminal ejecutará un bucle esperando los comandos del jugador (ej. `ver inventario`, `comprar armadura`, `ir calle 2`). Así, el jugador interactúa tranquilamente mientras el mundo sigue produciendo en segundo plano.

Para lograr la comunicación entre distintos ordenadores en las fases de Barrios y Pueblos, utilizaremos PyRO4 (Python Remote Objects). Esta librería permite que el hilo de un jugador en un ordenador puede invocar un método de un objeto alojado en el ordenador de su compañero (por ejemplo, mercado_remoto.comprar_recurso()) exactamente igual que si fuera un objeto local. Esto nos proporciona una transición limpia de la concurrencia local a un sistema completo.

Cuando el juego escale a Barrio o Pueblo y haya que conectar los distintos ordenadores, entrará en juego PyRO4. Cada calle levantará un hilo PyRO para escuchar peticiones en la red. Si un jugador viaja a otra calle y quiere comprar, su terminal accederá al inventario de la calle de destino de forma transparente. Por su parte, el Barrio Económico o Comercial será simplemente otro nodo expuesto por PyRO que, mediante un hilo secundario, revisará los inventarios globales periódicamente y ajustará los precios de forma sencilla según la oferta y la demanda y actualizará los precios y datos.

Dicho esto, vamos a crear el juego.

***AVISO:*** Se tiene en cuenta que todas las clases han sufrido miles de cambio a lo largo del desarrollo de las semanas debido a la inclusión de nuevos elementos y debido a las nuevas clases.

### Semana 1

#### Librerías

In [ ]:
# Importamos las librerías necesarias
from typing import Dict, List
from threading import Thread, Lock, Event
import time
from enum import Enum, auto
from abc import ABC, abstractmethod
import Pyro4

#### Clase Trabajador

In [ ]:

# Pasamos a crear la clase Trabajador y la clase que verifica su estado:
class EstadoTrabajador(Enum):
    INACTIVO = auto()
    PRODUCIENDO = auto() 
# Hay que tener en ceunta que Trabajador hereda de Thread
class Trabajador(Thread):
    # Iniciamos el constructor de clases
    def __init__(self,recetas:List[Dict],inventario:Dict[str,int],cerrojo:Lock, precios: dict[str,int]) -> None:
        super().__init__(daemon=True) # El trabajador desaparece cuando acaba el juego
        # Condición de recetas, para no saturar la memoria
        if len(recetas) > 3:
            raise ValueError(f" máximo 3 recetas permitidas.")
        # Definición de los atributos
        self.nombre = ""
        self._recetas = recetas
        self.inventario = inventario
        self.cerrojo = cerrojo
        self.precios = precios
        self.receta_activa = None
        self.estado = EstadoTrabajador.INACTIVO
        # Añadimos un último atributo para asegurar el flujo de los hilos
        self._activo = Event()
        self._activo.set()



# Definimos métodos privados para la correcta implementación del objeto trabajador
    # Método para comprobar si hay suficientes recursos para crear un objeto
    def _tiene_recursos(self, necesidades: dict[str, int]) -> bool:
        for recurso, cantidad in necesidades.items():
            if self.inventario.get(recurso, 0) < cantidad:
                return False
        return True
    # Método para reducir los recursos del almacen del trabajador
    def _consumir_recursos(self, necesidades: dict[str, int]) -> None:
        for recurso, cantidad in necesidades.items():
            self.inventario[recurso] -= cantidad



# Definimos los métodos públicos generals de la clase Trabajador:
    # Asignamos la tarea del hilo 
    def asignar_tarea(self, nombre_producto: str) -> None:
        for receta in self._recetas:
            if receta["produce"] == nombre_producto:
                self.receta_activa = receta
                self.estado = EstadoTrabajador.PRODUCIENDO
                print(f"{self.nombre} empieza a producir {nombre_producto}.")
                return
        print(f"{self.nombre} no puede producir '{nombre_producto}'.")
    # Paramos la tarea asignada
    def parar(self) -> None:
        self.estado = EstadoTrabajador.INACTIVO
        self.receta_activa = None
        print(f"{self.nombre} ha parado la producción.")
    # Método para fijar el precio de los items del trabajador
    def fijar_precio(self, producto: str, precio: int) -> None:
        if producto in self.precios:
            self.precios[producto] = precio
            print(f"El precio de {producto} ahora es {precio} monedas.")
        else:
            print(f"{self.nombre} no vende '{producto}'.")
    # Método para eliminar el stock del inventario del jugador
    def eliminar_stock(self, producto: str, cantidad: int) -> None:
        with self.cerrojo:
            if self.inventario.get(producto, 0) >= cantidad:
                self.inventario[producto] -= cantidad
                print(f"{self.nombre} elimina {cantidad} {producto} de su inventario.")
            else:
                print(f"No hay suficiente stock de {producto} para eliminar.")
    # Método para consultar las recetas del jugador. 
    def consultar_recetas(self) -> None:
        print(f"\n── Recetas de {self.nombre} ──")
        for receta in self._recetas:
            necesita = receta["necesita"] if receta["necesita"] else "nada"
            print(f"  {receta['produce']}: necesita {necesita}, tarda {receta['tiempo']}s")
    # Método para consultar los precios de los productos
    def consultar_precios(self) -> dict[str, int]:
        return dict(self.precios)
    #  Detenemos el hilo para limpiar el resultado final
    def detener(self) -> None:
        self._activo.clear()
        self.receta_activa = None
    # Método que crea todo el algoritmo de la clase trabajador.    
    def run(self) -> None:
        # Si el trabajador está en activo
        while self._activo.is_set():
            if self.receta_activa is None:
                time.sleep(0.5)
                continue
            receta = self.receta_activa
            # El trabajador produce
            with self.cerrojo:
                hay_recursos = self._tiene_recursos(receta["necesita"])
                if hay_recursos:
                    self._consumir_recursos(receta["necesita"])
            if not hay_recursos:
                time.sleep(0.5)
                continue
            print(f"{self.nombre} produciendo {receta['produce']}")
            time.sleep(receta["tiempo"])
            # Se notifica del item creado
            with self.cerrojo:
                self.inventario[receta["produce"]] += receta["cantidad"]
                print(f"[{self.nombre}] +{receta['cantidad']} {receta['produce']}.")
            # Esperamos
            time.sleep(1)            

Con esto habríamos terminado la implementación de trabajador y podríamos pasar a desarrollar el juego principal para finalizar la semana 1 de trabajo. Para ello vamos a desarrollar un bucle MUD, que es el hilo principal del juego. La idea con esto es iniciar el juego y que el jugador tenga control sobre su hilo. El juego se pone a correr de fondo y el usuario escribe en la terminal los comandos necesarios para jugar.

#### Vamos a desglosar como vamos a desarrollar el juego:

Cada participante del juego dispondrá de un personaje con nombre propio, un saldo de monedas y un estado que define qué está haciendo en cada momento: produciendo, atendiendo su tienda, viajando o descansando. Cada jugador tiene control exclusivo sobre su trabajador y por tanto sobre sus decisiones de producción, pero no sobre los de sus compañeros.

El juego se desarrolla en un bucle MUD que corre en el hilo principal de cada terminal. Mientras el jugador escribe comandos, su trabajador opera de forma autónoma en segundo plano. Los comandos disponibles permiten al jugador gestionar su producción, consultar el inventario de su calle, fijar precios, eliminar stock para reducir la oferta o simplemente descansar.

Cada calle dispone de una tienda que solo puede ser atendida por su propietario. Si un jugador de otra calle quiere comprar un producto, viaja hasta esa tienda y espera a ser atendido. Si el propietario se halla produciendo, le llega una notificación y decide si parar para atender al cliente o ignorarlo. Si transcurrido un tiempo el propietario no atiende, el cliente abandona la tienda. Una vez atendido, el comprador elige qué producto adquirir y en qué cantidad, limitado por su saldo de monedas. Del mismo modo, un jugador puede viajar a otra calle a ofrecer sus propios productos, fijando el precio que considere oportuno. El propietario de esa calle decidirá si acepta la oferta o la rechaza.

**IMPORTANTE!!!**

Cada jugador gestiona su propio saldo de monedas de forma independiente. Toda transacción económica actualiza los saldos de ambas partes de forma atómica, protegida por el cerrojo correspondiente, garantizando que no se produzcan condiciones de carrera en las operaciones de compraventa.

Dicho esto vamos a desarrollar las clases de juego:

#### Clase jugador

In [ ]:

# Pasamos a crear la clase Jugador y la clase que verifica su estado:
class EstadoJugador(Enum):
    INACTIVO = auto()
    PRODUCIENDO = auto()
    VIAJANDO = auto()
    DESCANSANDO = auto()
    EN_TIENDA = auto()



# Definimos la clase jugador
class Jugador:
    # Iniciamos el cosntructor de clases
    def __init__(self,nombre: str,trabajador: Trabajador,monedas: int = 50) -> None:
        self.nombre = nombre
        self.trabajador = trabajador
        self.monedas = monedas
        self.inventario: dict[str, int] = {}
        self.cerrojo = Lock()
        self.estado = EstadoJugador.DESCANSANDO
        self.localizacion:  str = "desconocida"
        self.trabajador.nombre = self.nombre
        self.precios_bolsa: dict[str,int] = {}



# Definimos los método privados de la clase
    # Definimos el método de añadir elementos al inventario del jugadro
    def _añadir_a_inventario(self, item: str, cantidad: int, precio_compra: int = 0) -> None:
        self.inventario[item] = self.inventario.get(item, 0) + cantidad
        # Fija el precio de reventa inicial al precio de compra si no existe
        if item not in self.precios_bolsa and precio_compra > 0:
            self.precios_bolsa[item] = precio_compra



# Definimos los métodos públicos de la clase
    # Definimos el método de compra de productos
    def comprar(self, item, cantidad, inventario_calle, cerrojo_calle, precio_unitario):
        coste_total = precio_unitario * cantidad
        with cerrojo_calle:
            if inventario_calle.get(item, 0) < cantidad:
                print(f"{self.nombre}: no hay suficiente {item} en la tienda.")
                return
            with self.cerrojo:
                if self.monedas < coste_total:
                    print(f"{self.nombre}: monedas insuficientes ({self.monedas}/{coste_total}).")
                    return
                inventario_calle[item] -= cantidad
                self.monedas -= coste_total
                self._añadir_a_inventario(item, cantidad, precio_unitario)  
        print(f"{self.nombre} compra {cantidad} x {item} por {coste_total} monedas.")
        print(f"  (precio de reventa inicial: {precio_unitario} mon/ud.)")
    # Definimos el método que fija el precio de la bolsa    
    def fijar_precio_bolsa(self, item: str, precio: int) -> None:
        """El jugador fija su propio precio de reventa para un item de su bolsa."""
        if item in self.inventario and self.inventario[item] > 0:
            self.precios_bolsa[item] = precio
            print(f"{self.nombre}: precio de reventa de {item} → {precio} monedas/ud.")
        else:
            print(f"{self.nombre} no tiene '{item}' en su bolsa.")
    # Definimos el método que hace una consulta a la bolsa
    def consultar_bolsa(self) -> None:
        with self.cerrojo:
            print(f"\n── Bolsa de {self.nombre} ({self.monedas} monedas) ──")
            if not self.inventario:
                print("  (vacía)")
            for item, cantidad in self.inventario.items():
                precio_rev = self.precios_bolsa.get(item, "sin precio")
                valor = precio_rev * cantidad if isinstance(precio_rev, int) else "?"
                print(f"  {item}: {cantidad} ud. | reventa: {precio_rev} mon/ud. | valor: {valor} mon.")
    # Definimos el método para vender productos
    def vender(self, item: str, cantidad: int, precio_unitario: int, comprador: "Jugador") -> None:
        coste_total = precio_unitario * cantidad
        # Primero el cerrojo del vendedor, luego el del comprador
        with self.cerrojo:
            if self.inventario.get(item, 0) < cantidad:
                print(f"{self.nombre}: no tienes suficiente {item} para vender.")
                return
            with comprador.cerrojo:
                if comprador.monedas < coste_total:
                    print(f"{comprador.nombre} no tiene monedas suficientes.")
                    return
                # Transacción completa
                self.inventario[item] -= cantidad
                self.monedas += coste_total
                comprador.monedas -= coste_total
                comprador._añadir_a_inventario(item, cantidad, precio_unitario)
        print(f"{self.nombre} vende {cantidad}x {item} a {comprador.nombre} por {coste_total} monedas.")


#### Clase Calle

In [ ]:

# Definimos la clase calle compuesta por dos jugadores
class Calle:
    # Definimos el constructor de clases
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self.jugadores: list[Jugador] = []
        self.inventario: dict[str, int] = {}
        self.cerrojo = Lock()


# Definimo los métodos privados de la clase calle
    # Método para actualizar el inventario de la calle, 
    # compuesto por el inventario de los dos jugadores que se encuentran en la calle
    def _actualizar_inventario(self, trabajador: Trabajador) -> None:
        for receta in trabajador._recetas:
            producto = receta["produce"]
            if producto not in self.inventario:
                self.inventario[producto] = 0
            for ingrediente in receta["necesita"]:
                if ingrediente not in self.inventario:
                    self.inventario[ingrediente] = 0


# Definimos los métodos públicos
    # Método para añadir un jugador a la calle 
    def añadir_jugador(self, jugador: Jugador) -> None:
        if len(self.jugadores) >= 2:
            raise ValueError(f"{self.nombre}: máximo 2 jugadores por calle.")
        # Asignamos el inventario y cerrojo de la calle al trabajador del jugador
        jugador.trabajador.inventario   = self.inventario
        jugador.trabajador.cerrojo = self.cerrojo
        jugador.localizacion = self.nombre
        self._actualizar_inventario(jugador.trabajador)
        self.jugadores.append(jugador)
        print(f"{jugador.nombre} se une a {self.nombre}.")
    # Método para iniciar todos los hilos de la calle
    def iniciar(self) -> None:
        for jugador in self.jugadores:
            jugador.trabajador.start()
        print(f"\n── {self.nombre} abierta ──")
    # Detenemos todos los procesos de la calle, deteniendo los hilos
    def detener(self) -> None:
        for jugador in self.jugadores:
            jugador.trabajador.detener()
        for jugador in self.jugadores:
            jugador.trabajador.join(timeout=10)  # ← no bloquea si el sleep es largo
        print(f"\n── {self.nombre} cerrada ──")
    # Método para consultar el inventario de la calle
    def consultar_inventario(self) -> None:
        with self.cerrojo:
            print(f"\n── Inventario de {self.nombre} ──")
            if not self.inventario:
                print(f"La calle {self.nombre} no tiene productos disponibles")
            for item, cantidad in self.inventario.items():
                print(f"  {item}: {cantidad}")
    # Método para consultar los jugadores que se hallan en la calle
    def consultar_jugadores(self) -> None:
        print(f"\n── Jugadores en {self.nombre} ──")
        for jugador in self.jugadores:
            print(f"  {jugador.nombre} | {jugador.estado.name} | {jugador.monedas} monedas")     
    # Método para swtichear entre jugadores
    def cambiar_jugador(self, nombre: str) -> Jugador | None:
        nombre = nombre.lower().strip("[]")
        for jugador in self.jugadores:
            if jugador.nombre.lower() == nombre:
                return jugador
        print(f"No existe ningún jugador llamado '{nombre}' en {self.nombre}.")
        return None       

#### MUD para la semana 1

In [ ]:
# Generamos el mud para que corra el juego
def mud(jugador_activo: Jugador, calle: Calle) -> None:
    # Mensaje de ayuda del juego con los distintos comandos
    ayuda = """
COMANDOS:

NO ES NECESARIO PONER COMILLAS NI SÍMBOLOS DE APERTURA Y CIERRE

producir [item]              → producir   
parar                        → parar   
inventario                   → ver calle  
bolsa                        → ver bolsa 
recetas                      → ver recetas
precios                      → ver precios
precio [item] [num]          → fijar precio
eliminar [item] [num]        → bajar stock
ayuda                        → comandos  
cambiar [jugador]            → cambiar PJ
salir                        → cerrar     
"""
    # Mensaje de inicio del juego
    print(f"\nBienvenido a {calle.nombre}, {jugador_activo.nombre}.")
    print(ayuda)
    # Se genera el bucle MUD
    while True:
        try:
            entrada = input(f"[{jugador_activo.nombre}] > ").strip().lower()
        except KeyboardInterrupt:
            print("\nInterrupción, cerrando la calle...")
            calle.detener()
            break
        # En el caso de que no haya inputs se prosigue
        if not entrada:
            continue
        partes = entrada.split()
        comando = partes[0]
        # Parte de la producción del MUD, para empezar a producir
        if comando == "producir":
            if len(partes) < 2:
                jugador_activo.trabajador.consultar_recetas()
                print("Uso: producir [item]")
            else:
                item = partes[1].strip("[]")
                jugador_activo.trabajador.asignar_tarea(item)
                jugador_activo.estado = EstadoJugador.PRODUCIENDO
        # Parte de la producción del MUD, para pararla concretamente
        elif comando == "parar":
            jugador_activo.trabajador.parar()
            jugador_activo.estado = EstadoJugador.DESCANSANDO
        # Parte de las consultas del MUD, para consultar el inventario
        elif comando == "inventario":
            calle.consultar_inventario()
        # Parte de las consultas del MUD, para consultar la bolsa
        elif comando == "bolsa":
            jugador_activo.consultar_bolsa()
        # Parte de las consultas del MUD, para consultar las recetas
        elif comando == "recetas":
            jugador_activo.trabajador.consultar_recetas()
        # Parte de las consultas del MUD, para consultar los precios
        elif comando == "precios":
            precios = jugador_activo.trabajador.consultar_precios()
            print(f"\n── Precios de {jugador_activo.trabajador.nombre} ──")
            for item, precio in precios.items():
                print(f"  {item}: {precio} monedas")
        # Parte de las peticiones del MUD, para fijar precios
        elif comando == "precio":
            if len(partes) < 3:
                print("Uso: precio [item] [cantidad]")
            else:
                try:
                    jugador_activo.trabajador.fijar_precio(partes[1].strip("[]"), int(partes[2]))
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para eliminar stock
        elif comando == "eliminar":
            if len(partes) < 3:
                print("Uso: eliminar [item] [cantidad]")
            else:
                try:
                    jugador_activo.trabajador.eliminar_stock(partes[1].strip("[]"), int(partes[2]))
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para recibir la ayuda
        elif comando == "ayuda":
            print(ayuda)
        # Parte de las peticiones del MUD, para salir del juego
        elif comando == "salir":
            print(f"\nHasta luego, {jugador_activo.nombre}.")
            calle.detener()
            break
        # Parte de las peticiones del MUD, para cambiar de jugador activo
        elif comando == "cambiar":
            if len(partes) < 2:
                print("Jugadores disponibles:")
                for j in calle.jugadores:
                    print(f"  {j.nombre} | {j.estado.name} | {j.monedas} monedas")
                print("Uso: cambiar [nombre]")
            else:
                resultado = calle.cambiar_jugador(partes[1])
                if resultado is not None:
                    jugador_activo = resultado
                    print(f"Ahora controlas a {jugador_activo.nombre}.")
        # Control de errores, para cuando el comando sea desconocido
        else:
            print(f"Comando '{comando}' desconocido. Escribe 'ayuda' para ver los comandos.")

#### Ejemplo generado con IA para comprobarlo

**IMPORTATE !!!**

Se recomienda exhaustivamente correr el ejemplo en la terminal con "P2_Implementación_Grupo_Q.py" como script para utilizar el ejemplo.

In [ ]:
# EJEMPLO PARA LA SEMANA 1
if __name__ == "__main__":


    # Recetas
    recetas_lenador = [
        {"produce": "madera", "cantidad": 1, "necesita": {}, "tiempo": 3}
    ]
    recetas_carpintero = [
        {"produce": "tablas",              "cantidad": 1, "necesita": {"madera": 2}, "tiempo": 4},
        {"produce": "muebles",             "cantidad": 1, "necesita": {"madera": 5}, "tiempo": 8},
        {"produce": "moldes_herramientas", "cantidad": 1, "necesita": {"madera": 1}, "tiempo": 2}
    ]


    # Trabajadores
    t_lenador    = Trabajador(    recetas_lenador,    {}, Lock(), {"madera": 3})
    t_carpintero = Trabajador( recetas_carpintero, {}, Lock(), {"tablas": 5, "muebles": 20, "moldes_herramientas": 8})


    # Jugadores
    marc  = Jugador("Marc",  t_lenador,    monedas=50)
    pedro = Jugador("Pedro", t_carpintero, monedas=50)


    # Calle
    calle = Calle("Calle del Ébano")
    calle.añadir_jugador(marc)
    calle.añadir_jugador(pedro)
    calle.iniciar()


    # Cada jugador entra al MUD en su propia terminal
    # Para probar en una sola máquina lanzamos el MUD con marc
    mud(marc, calle)

Con esto habremos finalizado la semana 1. Ahora tendríamos que pasar a crear la clase Barrio para la semana 2 y la clase Pueblo para la semana 3 respetando todo lo impuesto anteriormente, por ello mismo vamos a partir de las clases y objetos ya creados previamente. Todas las clases ya creadas están preparadas para el crecimiento del proyecto.

### Semana 2

Un barrio está compuesto por dos calles, es decir, cuatro estudiantes con cuatro profesiones. A diferencia de la calle, donde toda la comunicación ocurre en memoria compartida dentro de una misma máquina, el barrio introduce la programación distribuida: cada calle corre en un ordenador distinto y la comunicación entre ellas se realiza mediante invocaciones remotas usando PyRO4.

Existen dos tipos de barrio. El barrio comercial gestiona la distribución de recursos entre las dos calles que lo forman, implementando un mercado donde los jugadores pueden intercambiar productos entre calles distintas. El barrio económico gestiona la contabilidad monetaria del barrio, ajustando precios en función de la oferta y la demanda de cada calle de forma periódica y automática.

**IMPORTANTE!!!**

Cada calle levantará un servidor PyRO al iniciarse, exponiendo su inventario y sus métodos de compraventa a la red. El barrio será un nodo adicional que se conecta a ambas calles de forma remota. Un hilo secundario del barrio revisará periódicamente los inventarios y actualizará precios o redistribuirá recursos según su tipo. La comunicación remota es transparente: el jugador llama a mercado.comprar() exactamente igual que si fuera local.

#### Clase Barrio
Para definir los dos tipos de barrios de forma correcta vamos a definir una clase base ABC para implementar correctamente los métodos de los dos tipos de clases por ello, vamos a utilizar los métodos abstractos de la librería para crear las clases.

In [ ]:

# Clase base de barrio
# IMPORTANTE: En local comparten proceso, en distribuido cada Calle corre en una máquina distinta
class Barrio(ABC):
    # Iniciamos al constructor de clases
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self.calles: list[Calle] = []
        self.cerrojo = Lock()
        # Hilo secundario que corre la lógica propia del barrio (mercado o economía)
        self._hilo_barrio: Thread | None = None
        self._activo = Event()
        self._activo.set()
# Definimos los métodos privados
    # Definimos la lógica del barrio
    @abstractmethod
    def _logica_barrio(self) -> None:
        """
        BarrioComercial → redistribuye recursos.
        BarrioEconomico → ajusta precios.
        """
        pass
# Definimos los métodos públicos
    # Método para añadir una calle al barrio
    def añadir_calle(self, calle: Calle) -> None:
        if len(self.calles) >= 2:
            raise ValueError(f"{self.nombre}: máximo 2 calles por barrio.")
        self.calles.append(calle)
        print(f"{calle.nombre} se une al barrio {self.nombre}.")
    # Método para iniciar al barrio
    def iniciar(self) -> None:
        for calle in self.calles:
            calle.iniciar()
        # Arranca el hilo de lógica del barrio en segundo plano
        self._hilo_barrio = Thread(target=self._logica_barrio, daemon=True)
        self._hilo_barrio.start()
        print(f"\n── {self.nombre} abierto ──")
    # Detiene el barrio y por tanto las dos calles y por tanto los 4 trabajadores
    def detener(self) -> None:
        self._activo.clear()
        for calle in self.calles:
            calle.detener()
        if self._hilo_barrio:
            self._hilo_barrio.join(timeout=5)
        print(f"\n── {self.nombre} cerrado ──")
    # Método para consultar el estado del barrio
    def consultar_estado(self) -> None:
        print(f"\n── Estado del barrio {self.nombre} ──")
        for calle in self.calles:
            calle.consultar_inventario()
            calle.consultar_jugadores()
    # Método para consultar la contabilidad de todo el conjunto del barrio
    def consultar_contabilidad(self) -> None:
        print(f"\n── Contabilidad de {self.nombre} ──")
        for calle in self.calles:
            print(f"  {calle.nombre}:")
            for jugador in calle.jugadores:
                print(f"    {jugador.nombre}: {jugador.monedas} monedas")
    # Método para hacer un ranking de las calles viendo cual es la que más genera por barrio
    def ranking_calles(self) -> None:
        """
        Muestra las calles del barrio ordenadas por la riqueza total
        de sus jugadores (monedas acumuladas).
        """
        ranking = []
        for calle in self.calles:
            total = sum(jugador.monedas for jugador in calle.jugadores)
            ranking.append((calle.nombre, total))
        ranking.sort(key=lambda x: x[1], reverse=True)
        print(f"\n── Ranking de calles — {self.nombre} ──")
        for posicion, (nombre_calle, total) in enumerate(ranking, start=1):
            print(f"  {posicion}. {nombre_calle}: {total} monedas en total")
    # Obtenemos el total de monedas del barrio, sumando las monedas de los 4 personas de los que forman parte
    def total_monedas(self) -> None:
        total_barrio = 0
        print(f"\n── Riqueza total de {self.nombre} ──")
        for calle in self.calles:
            total_calle = sum(jugador.monedas for jugador in calle.jugadores)
            total_barrio += total_calle
            print(f"  {calle.nombre}: {total_calle} monedas")
            for jugador in calle.jugadores:
                print(f"    {jugador.nombre}: {jugador.monedas} monedas")
        print(f"  ────────────────────────")
        print(f"  TOTAL BARRIO: {total_barrio} monedas")
    # Método para cambiar jugadores entre las distintas calles
    def cambiar_jugador(self, nombre: str) -> "tuple[Jugador, Calle] | None":
        nombre = nombre.lower().strip("[]")
        for calle in self.calles:
            for jugador in calle.jugadores:
                if jugador.nombre.lower() == nombre:
                    return jugador, calle
        print(f"No existe ningún jugador llamado '{nombre}' en {self.nombre}.")
        return None
    # Método para obtener precios
    def obtener_precio(self, item: str, calle_origen: "Calle") -> int | None:
        """Devuelve el precio vigente del item en la calle origen."""
        for jugador in calle_origen.jugadores:
            if item in jugador.trabajador.precios:
                return jugador.trabajador.precios[item]
        return None
    # Método para ver todo el almacenamiento del barrio
    def consultar_almacen(self) -> None:
        almacen: dict[str, int] = {}
        for calle in self.calles:
            with calle.cerrojo:
                for item, cantidad in calle.inventario.items():
                    almacen[item] = almacen.get(item, 0) + cantidad
        print(f"\n── Almacén de {self.nombre} ──")
        if not almacen:
            print("  (vacío)")
        else:
            for item, cantidad in almacen.items():
                # Desglose por calle debajo de cada item
                print(f"  {item}: {cantidad} (total)")
                for calle in self.calles:
                    parcial = calle.inventario.get(item, 0)
                    if parcial > 0:
                        print(f"    {calle.nombre}: {parcial}")


#### Barrio comercial

El barrio tiene la tarea de según los objetos del inventario transfiere recursos entre las calles para que no haya desequilibrios. Tiene la finalidad de gestionar los productos.

In [ ]:
# Iniciamos la clase Barrio Comercial
class BarrioComercial(Barrio):
    # Iniciamos el constructor de clases
    def __init__(self, nombre: str, intervalo: int = 15) -> None:
        super().__init__(nombre)
        self.intervalo = intervalo   # umbral de segundos para el barrio
        self.umbral_exceso: int = 5
        self.precios_mercado: dict[str, int] = {}   
# Definimos los métodos privados
    # Definimos la lógica del barrio
    def _logica_barrio(self) -> None:
        while self._activo.is_set():
            time.sleep(self.intervalo)
            self._redistribuir_recursos()
    # Método para redistribuir los recursos entre las distintas calles
    def _redistribuir_recursos(self) -> None:
        if len(self.calles) < 2:
            return
        calle_a, calle_b = self.calles
        # Abrimos todos los cerrojos
        with self.cerrojo:
            with calle_a.cerrojo:
                with calle_b.cerrojo:
                    # Snapshot para que los dos bucles lean el estado inicial
                    snap_a = dict(calle_a.inventario)
                    snap_b = dict(calle_b.inventario)
                    # A -> B
                    for item in snap_a:
                        exceso_a = snap_a[item] - self.umbral_exceso
                        deficit_b = self.umbral_exceso - snap_b.get(item, 0)
                        if exceso_a > 0 and deficit_b > 0:
                            transferir = min(exceso_a, deficit_b)
                            calle_a.inventario[item] -= transferir
                            calle_b.inventario[item] = calle_b.inventario.get(item, 0) + transferir
                            print(f"[Mercado {self.nombre}] {transferir} {item}: "
                                  f"{calle_a.nombre} → {calle_b.nombre}.")
                    # B -> A
                    for item in snap_b:
                        exceso_b = snap_b[item] - self.umbral_exceso
                        deficit_a = self.umbral_exceso - snap_a.get(item, 0)
                        if exceso_b > 0 and deficit_a > 0:
                            transferir = min(exceso_b, deficit_a)
                            calle_b.inventario[item] -= transferir
                            calle_a.inventario[item] = calle_a.inventario.get(item, 0) + transferir
                            print(f"[Mercado {self.nombre}] {transferir} {item}: "
                                  f"{calle_b.nombre} → {calle_a.nombre}.")

    # Método para consultar el mercado y los distintos objetos que tiene el barrio
    def consultar_mercado(self) -> None:
        print(f"\n── Mercado de {self.nombre} (umbral exceso: {self.umbral_exceso}) ──")
        for calle in self.calles:
            calle.consultar_inventario()
    # Método para fijar los precios del mercado del barrio
    def fijar_precio_mercado(self, item: str, precio: int) -> None:
        """Fija un precio centralizado de mercado para un item."""
        self.precios_mercado[item] = precio
        print(f"[Mercado {self.nombre}] Precio de mercado de {item}: {precio} monedas.")
    # Método para obtener los precios del mercado del barrio
    def obtener_precio(self, item: str, calle_origen: "Calle") -> int | None:
        """Precio de mercado tiene prioridad. Si no existe, usa el del trabajador."""
        if item in self.precios_mercado:
            return self.precios_mercado[item]
        return super().obtener_precio(item, calle_origen)


#### Barrio económico
Este barrio gestiona y marca los precios de las dos calles a las que está ligada el barrio. Tiene la finalidad de gestionar la contabilidad del barrio.

In [ ]:

# Iniciamos la clase Barrio Económico            
class BarrioEconomico(Barrio):
    """
    Cada N segundos ajusta precios según oferta:
      - stock > umbral_alto  → baja precio (exceso de oferta)
      - stock < umbral_bajo  → sube precio (escasez)
    """
    # Definimos el constructor de clases
    def __init__(self, nombre: str, intervalo: int = 20) -> None:
        super().__init__(nombre)
        self.intervalo = intervalo
        self.umbral_alto: int  = 10   # más de 10 → baja precio
        self.umbral_bajo: int  = 2    # menos de 2 → sube precio
        self.variacion: float  = 0.2  # 20% de variación por ciclo
# Definimos los métodos privados
    # Definimos la lógica económica del barrio
    def _logica_barrio(self) -> None:
        while self._activo.is_set():
            time.sleep(self.intervalo)
            self._ajustar_precios()
    # Ajustamos el precio de los productos incluidos en el barrio
    def _ajustar_precios(self) -> None:
        # Abrimos los cerrojos y modificamos los precios
        for calle in self.calles:
            with calle.cerrojo:
                for jugador in calle.jugadores:
                    for item, precio_actual in jugador.trabajador.precios.items():
                        stock = calle.inventario.get(item, 0)
                        if stock > self.umbral_alto:
                            nuevo_precio = max(1, int(precio_actual * (1 - self.variacion)))
                            jugador.trabajador.precios[item] = nuevo_precio
                            print(
                                f"[Economía {self.nombre}] "
                                f"Exceso de {item} ({stock} uds.) → "
                                f"precio {precio_actual} → {nuevo_precio} monedas."
                            )
                        elif stock < self.umbral_bajo:
                            nuevo_precio = int(precio_actual * (1 + self.variacion))
                            jugador.trabajador.precios[item] = nuevo_precio
                            print(
                                f"[Economía {self.nombre}] "
                                f"Escasez de {item} ({stock} uds.) → "
                                f"precio {precio_actual} → {nuevo_precio} monedas."
                            )


#### Actualizamos el MUD para la semana 2

Implementamos los nuevos métodos definidos en el barrio

In [ ]:

# Definimos el nuevo mud
# Generamos el mud para que corra el juego
def mud(jugador_activo: Jugador, calle: Calle, barrio:Barrio | None = None) -> None:
    # Mensaje de ayuda del juego con los distintos comandos
    ayuda = """
COMANDOS:


NO ES NECESARIO PONER COMILLAS NI SÍMBOLOS DE APERTURA Y CIERRE


producir [item]                        → empezar a producir
parar                                  → parar producción
inventario                             → ver inventario de la calle
bolsa                                  → ver tu bolsa y monedas
recetas                                → ver tus recetas
precios                                → ver tus precios de producción
precio [item] [num]                    → fijar precio de producción
precio_bolsa [item] [num]              → fijar precio de reventa personal
comprar [item] [num]                   → comprar item de la calle
vender [item] [num] [jugador]          → vender desde tu bolsa (pide confirmación)
depositar [item] [num]                 → depositar item de la bolsa al taller
retirar [item] [num]                   → retirar item del taller a la bolsa
eliminar [item] [num]                  → eliminar stock de la calle
almacen                                → ver almacén del barrio
cambiar [jugador]                      → cambiar de jugador activo
ranking                                → ranking de calles por riqueza
riqueza                                → riqueza total del barrio
ayuda                                  → mostrar este menú
salir                                  → cerrar el juego
"""
    # Mensaje de inicio del juego
    print(f"\nBienvenido a {calle.nombre}, {jugador_activo.nombre}.")
    print(ayuda)
    # Se genera el bucle MUD
    while True:
        try:
            entrada = input(f"[{jugador_activo.nombre}] > ").strip().lower()
        except KeyboardInterrupt:
            print("\nInterrupción, cerrando...")
            if barrio:
                barrio.detener()
            else:
                calle.detener()
            break
        # En el caso de que no haya inputs se prosigue
        if not entrada:
            continue
        partes = entrada.split()
        comando = partes[0]
        # Parte de la producción del MUD, para empezar a producir
        if comando == "producir":
            if len(partes) < 2:
                jugador_activo.trabajador.consultar_recetas()
                print("Uso: producir [item]")
            else:
                item = partes[1].strip("[]")
                jugador_activo.trabajador.asignar_tarea(item)
                jugador_activo.estado = EstadoJugador.PRODUCIENDO
        # Parte de la producción del MUD, para pararla concretamente
        elif comando == "parar":
            jugador_activo.trabajador.parar()
            jugador_activo.estado = EstadoJugador.DESCANSANDO
        # Parte de las consultas del MUD, para consultar el inventario
        elif comando == "inventario":
            calle.consultar_inventario()
        # Parte de las consultas del MUD, para consultar la bolsa
        elif comando == "bolsa":
            jugador_activo.consultar_bolsa()
        # Parte de las consultas del MUD, para consultar las recetas
        elif comando == "recetas":
            jugador_activo.trabajador.consultar_recetas()
        # Parte de las consultas del MUD, para consultar los precios
        elif comando == "precios":
            precios = jugador_activo.trabajador.consultar_precios()
            print(f"\n── Precios de {jugador_activo.trabajador.nombre} ──")
            for item, precio in precios.items():
                print(f"  {item}: {precio} monedas")
        # Parte de las peticiones del MUD, para fijar precios
        elif comando == "precio":
            if len(partes) < 3:
                print("Uso: precio [item] [cantidad]")
            else:
                try:
                    jugador_activo.trabajador.fijar_precio(partes[1].strip("[]"), int(partes[2]))
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para comprar elementos
        elif comando == "comprar":
            if len(partes) < 3:
                print("Uso: comprar [item] [cantidad]")
            else:
                try:
                    item_c = partes[1].strip("[]")
                    cant_c = int(partes[2])

                    calle_origen = None

                    # 1. Busca stock en la calle actual
                    if calle.inventario.get(item_c, 0) >= cant_c:
                        calle_origen = calle

                    # 2. Si no, busca en el resto del barrio
                    if calle_origen is None and barrio:
                        for c in barrio.calles:
                            if c is calle:
                                continue
                            if c.inventario.get(item_c, 0) >= cant_c:
                                calle_origen = c
                                break

                    if calle_origen is None:
                        print(f"No hay suficiente '{item_c}' disponible en el barrio.")
                    else:
                        # Obtiene el precio según el tipo de barrio
                        if barrio:
                            precio_u = barrio.obtener_precio(item_c, calle_origen)
                        else:
                            precio_u = None
                            for j in calle_origen.jugadores:
                                if item_c in j.trabajador.precios:
                                    precio_u = j.trabajador.precios[item_c]
                                    break

                        if precio_u is None:
                            print(f"No hay precio fijado para '{item_c}'.")
                        else:
                            if calle_origen is not calle:
                                print(f"(Comprando de {calle_origen.nombre})")
                            jugador_activo.comprar(item_c, cant_c,
                                                   calle_origen.inventario,
                                                   calle_origen.cerrojo,
                                                   precio_u)
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para vender elementos
        elif comando == "vender":
            if len(partes) < 4:
                print("Uso: vender [item] [cantidad] [comprador]")
            else:
                try:
                    item_v   = partes[1].strip("[]")
                    cant_v   = int(partes[2])
                    nombre_c = partes[3].strip("[]")

                    comprador_obj = None
                    if barrio:
                        res = barrio.cambiar_jugador(nombre_c)
                        if res:
                            comprador_obj = res[0]
                    else:
                        comprador_obj = calle.cambiar_jugador(nombre_c)

                    if comprador_obj is None:
                        print(f"No se encontró al jugador '{nombre_c}'.")
                    elif comprador_obj is jugador_activo:
                        print("No puedes venderte a ti mismo.")
                    else:
                        # Precio de bolsa tiene prioridad sobre precio del trabajador
                        precio_u = jugador_activo.precios_bolsa.get(item_v) \
                                   or jugador_activo.trabajador.precios.get(item_v)
                        if precio_u is None:
                            print(f"Sin precio para '{item_v}'. Usa: precio_bolsa {item_v} [num]")
                        else:
                            coste = precio_u * cant_v
                            print(f"\n{jugador_activo.nombre} ofrece {cant_v} x {item_v} "
                                  f"a {comprador_obj.nombre} por {coste} monedas.")
                            # ── Confirmación del comprador ──
                            try:
                                resp = input(f"[{comprador_obj.nombre}] ¿Aceptas? (s/n) > ").strip().lower()
                            except KeyboardInterrupt:
                                resp = "n"
                            if resp == "s":
                                jugador_activo.vender(item_v, cant_v, precio_u, comprador_obj)
                            else:
                                print(f"{comprador_obj.nombre} rechazó la oferta.")
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")    
        # Parte de las peticiones del MUD, para depositar items de la bolsa al taller
        elif comando == "depositar":
            if len(partes) < 3:
                print("Uso: depositar [item] [cantidad]")
            else:
                try:
                    item_d = partes[1].strip("[]")
                    cant_d = int(partes[2])
                    with jugador_activo.cerrojo:
                        if jugador_activo.inventario.get(item_d, 0) < cant_d:
                            print(f"No tienes suficiente {item_d} en la bolsa.")
                        else:
                            with calle.cerrojo:
                                jugador_activo.inventario[item_d] -= cant_d
                                calle.inventario[item_d] = calle.inventario.get(item_d, 0) + cant_d
                            print(f"Has depositado {cant_d} {item_d} en el taller.")
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para retirar items del taller a la bolsa
        elif comando == "retirar":
            if len(partes) < 3:
                print("Uso: retirar [item] [cantidad]")
            else:
                try:
                    item_r = partes[1].strip("[]")
                    cant_r = int(partes[2])
                    with calle.cerrojo:
                        if calle.inventario.get(item_r, 0) < cant_r:
                            print(f"No hay suficiente {item_r} en el taller.")
                        else:
                            with jugador_activo.cerrojo:
                                calle.inventario[item_r] -= cant_r
                                jugador_activo._añadir_a_inventario(item_r, cant_r)
                            print(f"Has retirado {cant_r} {item_r} a tu bolsa.")
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para eliminar stock
        elif comando == "eliminar":
            if len(partes) < 3:
                print("Uso: eliminar [item] [cantidad]")
            else:
                try:
                    jugador_activo.trabajador.eliminar_stock(partes[1].strip("[]"), int(partes[2]))
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para recibir la ayuda
        elif comando == "ayuda":
            print(ayuda)
        # Parte de las peticiones del MUD, para salir del juego
        elif comando == "salir":
            print(f"\nHasta luego, {jugador_activo.nombre}.")
            if barrio:
                barrio.detener()
            else:
                calle.detener()
            break
        # Parte de las peticiones del MUD, para cambiar de jugador activo
        elif comando == "cambiar":
            if len(partes) < 2:
                # Lista todos los jugadores: del barrio si existe, si no de la calle
                if barrio:
                    print("Jugadores disponibles en el barrio:")
                    for c in barrio.calles:
                        marca = " ◄ (aquí)" if c.nombre == calle.nombre else ""
                        print(f"  [{c.nombre}]{marca}")
                        for j in c.jugadores:
                            print(f"    {j.nombre} | {j.estado.name} | {j.monedas} monedas")
                else:
                    print("Jugadores disponibles:")
                    for j in calle.jugadores:
                        print(f"  {j.nombre} | {j.estado.name} | {j.monedas} monedas")
                print("Uso: cambiar [nombre]")
            else:
                if barrio:
                    resultado = barrio.cambiar_jugador(partes[1])
                    if resultado is not None:
                        jugador_activo, calle = resultado   # ← actualiza AMBAS
                        print(f"Ahora controlas a {jugador_activo.nombre} "
                              f"en {calle.nombre}.")
                else:
                    resultado = calle.cambiar_jugador(partes[1])
                    if resultado is not None:
                        jugador_activo = resultado
                        print(f"Ahora controlas a {jugador_activo.nombre}.")
        # Parte de las consultas del MUD, para ver el ranking de calles
        elif comando == "ranking":
            if barrio:
                barrio.ranking_calles()
            else:
                print("No estás en un barrio todavía.")
        # Parte de las consultas del MUD, para ver el precio de la bolsa
        elif comando == "precio_bolsa":
            if len(partes) < 3:
                print("Uso: precio_bolsa [item] [num]")
            else:
                try:
                    jugador_activo.fijar_precio_bolsa(partes[1].strip("[]"), int(partes[2]))
                except ValueError:
                    print("El precio tiene que ser un número entero.")
        # Parte de las consultas del MUD, para ver la riqueza total del barrio
        elif comando == "riqueza":
            if barrio:
                barrio.total_monedas()
            else:
                jugador_activo.consultar_bolsa()
        # Parte de las consultas del MUD, para ver todos los objetos de un barrio
        elif comando == "almacen":
            if barrio:
                barrio.consultar_almacen()
            else:
                print("No estás en un barrio todavía.")
        # Control de errores, para cuando el comando sea desconocido
        else:
            print(f"Comando '{comando}' desconocido. Escribe 'ayuda' para ver los comandos.")


#### Ejemplo desarrollado con IA para la implementación de la semana 2

**IMPORTATE !!!**

Se recomienda exhaustivamente correr el ejemplo en la terminal con "P2_Implementación_Grupo_Q.py" como script para utilizar el ejemplo.

In [ ]:
# EJEMPLO PARA LA SEMANA 2            
if __name__ == "__main__":


    # ── Calle del Ébano (madera/carpintería) ──
    recetas_lenador = [
        {"produce": "madera", "cantidad": 1, "necesita": {}, "tiempo": 3}
    ]
    recetas_carpintero = [
        {"produce": "tablas",  "cantidad": 1, "necesita": {"madera": 2}, "tiempo": 4},
        {"produce": "muebles", "cantidad": 1, "necesita": {"madera": 5}, "tiempo": 8},
    ]
    t_lenador    = Trabajador(recetas_lenador,    {}, Lock(), {"madera": 3})
    t_carpintero = Trabajador(recetas_carpintero, {}, Lock(), {"tablas": 5, "muebles": 20})
    marc  = Jugador("Marc",  t_lenador,    monedas=50)
    pedro = Jugador("Pedro", t_carpintero, monedas=50)
    calle_ebano = Calle("Calle del Ébano")
    calle_ebano.añadir_jugador(marc)
    calle_ebano.añadir_jugador(pedro)


    # ── Calle del Hierro (minería/herrería) ──
    recetas_minero = [
        {"produce": "mineral", "cantidad": 1, "necesita": {}, "tiempo": 4}
    ]
    recetas_herrero = [
        {"produce": "hierro",  "cantidad": 1, "necesita": {"mineral": 2}, "tiempo": 5},
        {"produce": "herramientas", "cantidad": 1, "necesita": {"hierro": 1}, "tiempo": 6},
    ]
    t_minero  = Trabajador(recetas_minero,  {}, Lock(), {"mineral": 2})
    t_herrero = Trabajador(recetas_herrero, {}, Lock(), {"hierro": 8, "herramientas": 15})
    ana    = Jugador("Ana",    t_minero,  monedas=50)
    carlos = Jugador("Carlos", t_herrero, monedas=50)
    calle_hierro = Calle("Calle del Hierro")
    calle_hierro.añadir_jugador(ana)
    calle_hierro.añadir_jugador(carlos)


    # ── Barrio Comercial ──
    barrio = BarrioComercial("Barrio del Mercado", intervalo=15)
    barrio.añadir_calle(calle_ebano)
    barrio.añadir_calle(calle_hierro)
    barrio.iniciar()


    # ── MUD: Marc arranca el juego ──
    mud(marc, calle_ebano, barrio)


### Semana 3

Una vez ya hemos definido el personaje, el hilo trabajador, las calles, los barrios falta el pueblo en su conjunto que recopila todo esto anteriormente mencionado. Para implementar el pueblo vamos a tener en cuenta todo el código ya construido y vamos a desarrollarlo de forma que cumpla con los requisitos impuestos por la práctica. Y para ello vamos a implementar PyRO4 para hacer el videojuego un videojuego de verdad para que tena una distribuición propia de un proyecto de este estilo.

**IMPORTANTE!!!**

La librería PyRO4 no se ha implementado anteriormente, ya que vamos a modificar nuevamente las clases anteriores se recuerda que se ejecute en la terminal para una correcta implementación el script adjunto al notebook.

Como vamos a implementar una nueva librería vamos a recrear las clases a partir de la semana 3 incluyendo los métodos básicos para su implementación. 

**FUNCIONAMIENTO DE PYRO 4:**
Rquiere un nameserver activo en una terminal aparte


#### Calle remota y su registro

Creamos la clase calle pero incorporada al server, ahora todas las clases deben de ser activades por el nameserver. Debido a que es una librería de la que desconocíamos de su existencia vamos a intentar poner el máximo de enunciados posibles así como instrucciones.

In [ ]:
@Pyro4.expose
class CalleRemota:
    # Iniciamos el constructor de clases
    def __init__(self, calle: "Calle") -> None:
        self._calle = calle

    # Devuelve una copia del inventario (snapshot seguro)
    def get_inventario(self) -> dict:
        with self._calle.cerrojo:
            return dict(self._calle.inventario)

    # Transfiere una cantidad de un item al inventario de esta calle
    def transferir(self, item: str, cantidad: int) -> bool:
        with self._calle.cerrojo:
            self._calle.inventario[item] = self._calle.inventario.get(item, 0) + cantidad
            return True

    # Retira una cantidad de un item del inventario de esta calle
    def retirar(self, item: str, cantidad: int) -> bool:
        with self._calle.cerrojo:
            if self._calle.inventario.get(item, 0) < cantidad:
                return False
            self._calle.inventario[item] -= cantidad
            return True

    # Devuelve info básica de los jugadores (sin objetos no serializables)
    def get_jugadores_info(self) -> list:
        return [
            {
                "nombre": j.nombre,
                "monedas": j.monedas,
                "estado": j.estado.name,
                "precios": dict(j.trabajador.precios)
            }
            for j in self._calle.jugadores
        ]

    # Devuelve el nombre de la calle
    def get_nombre(self) -> str:
        return self._calle.nombre

def registrar_calle_pyro(calle: "Calle") -> None:
    def _servidor():
        # Creamos el objeto remoto envolviendo la calle local
        remota = CalleRemota(calle)
        with Pyro4.Daemon() as daemon:
            # Registramos en el nameserver con el nombre de la calle
            with Pyro4.locateNS() as ns:
                uri = daemon.register(remota)
                ns.register(calle.nombre, uri)
                print(f"[PyRO4] {calle.nombre} registrada → {uri}")
            # Escucha peticiones indefinidamente (hilo daemon)
            daemon.requestLoop()

    hilo = Thread(target=_servidor, daemon=True)
    hilo.start()

# Creamos un Proxy a una calle remota
# El barrio la usa para conectarse a calles en otras máquinas.
def obtener_proxy_calle(nombre_calle: str) -> "Pyro4.Proxy":
    with Pyro4.locateNS() as ns:
        uri = ns.lookup(nombre_calle)
    return Pyro4.Proxy(uri)

# Creamos su medio distribuido
# Se sobreescribe iniciar() para levantar el servidor PyRO4
# cuando distribuido=True. El resto de la clase no cambia.
_iniciar_original = Calle.iniciar

def _iniciar_distribuido(self, distribuido: bool = False) -> None:
    _iniciar_original(self)
    if distribuido:
        registrar_calle_pyro(self)

Calle.iniciar = _iniciar_distribuido

#### Barrios distribuidos y su implementación
En este caso modificamos la clase barrio para que los barrios sean capaces de soportar el modo distribuido. Para ello, vamos a implementar nuevamente proxys en la creación de las nuevas clases

In [ ]:
# REDEFINICIÓN DE CLASE BARRIO
_init_barrio_original = Barrio.__init__

def _init_barrio_nuevo(self, nombre: str, distribuido: bool = False) -> None:
    _init_barrio_original(self, nombre)
    self.distribuido = distribuido
    # Para el modo distribuido usamos nombres de calles remotas para crear Proxies
    self._nombres_calles_remotas = []
Barrio.__init__ = _init_barrio_nuevo

def _añadir_calle_distribuido(self, calle) -> None:
    if self.distribuido:
        # En distribuido recibimos el nombre (str) de la calle remota
        if isinstance(calle, str):
            if len(self._nombres_calles_remotas) >= 2:
                raise ValueError(f"{self.nombre}: máximo 2 calles por barrio.")
            self._nombres_calles_remotas.append(calle)
            print(f"Calle remota '{calle}' añadida al barrio {self.nombre}.")
        else:
            raise TypeError("En modo distribuido añadir_calle() recibe el nombre (str) de la calle.")
    else:
        # Modo local: comportamiento original
        if len(self.calles) >= 2:
            raise ValueError(f"{self.nombre}: máximo 2 calles por barrio.")
        self.calles.append(calle)
        print(f"{calle.nombre} se une al barrio {self.nombre}.")
Barrio.añadir_calle = _añadir_calle_distribuido

# Método para obtener el inventario de una calle (local o remota)
def _get_inventario_calle(self, calle_o_nombre) -> dict:
    if self.distribuido:
        proxy = obtener_proxy_calle(calle_o_nombre)
        return proxy.get_inventario()
    else:
        with calle_o_nombre.cerrojo:
            return dict(calle_o_nombre.inventario)
Barrio._get_inventario_calle = _get_inventario_calle

# Método para transferir recursos a una calle (local o remota)
def _transferir_a_calle(self, calle_o_nombre, item: str, cantidad: int) -> None:
    if self.distribuido:
        proxy = obtener_proxy_calle(calle_o_nombre)
        proxy.transferir(item, cantidad)
    else:
        with calle_o_nombre.cerrojo:
            calle_o_nombre.inventario[item] = calle_o_nombre.inventario.get(item, 0) + cantidad
Barrio._transferir_a_calle = _transferir_a_calle


# Método para retirar recursos de una calle (local o remota)
def _retirar_de_calle(self, calle_o_nombre, item: str, cantidad: int) -> bool:
    if self.distribuido:
        proxy = obtener_proxy_calle(calle_o_nombre)
        return proxy.retirar(item, cantidad)
    else:
        with calle_o_nombre.cerrojo:
            if calle_o_nombre.inventario.get(item, 0) < cantidad:
                return False
            calle_o_nombre.inventario[item] -= cantidad
            return True
Barrio._retirar_de_calle = _retirar_de_calle

# REDEFINICIÓN DE CLASE BARRIO COMERCIAL
_redistribuir_original = BarrioComercial._redistribuir_recursos

def _redistribuir_distribuido(self) -> None:
    if not self.distribuido:
        # Modo local: lógica original (ya corregida)
        _redistribuir_original(self)
        return
    # Modo distribuido: usamos Proxies
    if len(self._nombres_calles_remotas) < 2:
        return
    nombre_a, nombre_b = self._nombres_calles_remotas
    snap_a = self._get_inventario_calle(nombre_a)
    snap_b = self._get_inventario_calle(nombre_b)
    # A -> B
    for item in snap_a:
        exceso_a = snap_a[item] - self.umbral_exceso
        deficit_b = self.umbral_exceso - snap_b.get(item, 0)
        if exceso_a > 0 and deficit_b > 0:
            transferir = min(exceso_a, deficit_b)
            retirado = self._retirar_de_calle(nombre_a, item, transferir)
            if retirado:
                self._transferir_a_calle(nombre_b, item, transferir)
                print(f"[Mercado {self.nombre}] {transferir} {item}: {nombre_a} → {nombre_b}.")
    # B -> A
    for item in snap_b:
        exceso_b = snap_b[item] - self.umbral_exceso
        deficit_a = self.umbral_exceso - snap_a.get(item, 0)
        if exceso_b > 0 and deficit_a > 0:
            transferir = min(exceso_b, deficit_a)
            retirado = self._retirar_de_calle(nombre_b, item, transferir)
            if retirado:
                self._transferir_a_calle(nombre_a, item, transferir)
                print(f"[Mercado {self.nombre}] {transferir} {item}: {nombre_b} → {nombre_a}.")
BarrioComercial._redistribuir_recursos = _redistribuir_distribuido

# REDEFINICIÓN DE BARRIO ECONÓMICO
_ajustar_original = BarrioEconomico._ajustar_precios

def _ajustar_distribuido(self) -> None:
    if not self.distribuido:
        _ajustar_original(self)
        return
    # Modo distribuido: leemos info de jugadores via Proxy
    for nombre_calle in self._nombres_calles_remotas:
        proxy = obtener_proxy_calle(nombre_calle)
        inventario = proxy.get_inventario()
        jugadores_info = proxy.get_jugadores_info()
        for info in jugadores_info:
            for item, precio_actual in info["precios"].items():
                stock = inventario.get(item, 0)
                if stock > self.umbral_alto:
                    nuevo_precio = max(1, int(precio_actual * (1 - self.variacion)))
                    print(
                        f"[Economía {self.nombre}] "
                        f"Exceso de {item} ({stock} uds.) → "
                        f"precio {precio_actual} → {nuevo_precio} monedas. "
                        f"[Notifica a {info['nombre']} en {nombre_calle}]"
                    )
                elif stock < self.umbral_bajo:
                    nuevo_precio = int(precio_actual * (1 + self.variacion))
                    print(
                        f"[Economía {self.nombre}] "
                        f"Escasez de {item} ({stock} uds.) → "
                        f"precio {precio_actual} → {nuevo_precio} monedas. "
                        f"[Notifica a {info['nombre']} en {nombre_calle}]"
                    )
BarrioEconomico._ajustar_precios = _ajustar_distribuido

#### PUEBLO

Para finalizar las implementaciones de clases y objetos vamos a desarrollar la clase pueblo teniendo en cuenta lo anterior definido. Qué es un conjunto de dos barrios, uno comercial y uno económico. A su vez, cada barrio dispone de dos calles, y a su vez cada calle dos trabajadores. Con esto formamos un pueblo de 8 trabajadores.

In [ ]:
class Pueblo:
    # Iniciamos el constructor de clases
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self.barrios: list[Barrio] = []
        self.cerrojo = Lock()
        # Hilo secundario que gestiona la lógica inter-barrios
        self._hilo_pueblo: Thread | None = None
        self._activo = Event()
        self._activo.set()
        self.intervalo: int = 30   # segundos entre ciclos de lógica del pueblo

# Definimos los métodos privados
    # Lógica de oferta/demanda global entre los dos barrios del pueblo
    def _logica_pueblo(self) -> None:
        while self._activo.is_set():
            time.sleep(self.intervalo)
            self._equilibrar_barrios()

    # Equilibra recursos entre los dos barrios
    def _equilibrar_barrios(self) -> None:
        if len(self.barrios) < 2:
            return
        barrio_a, barrio_b = self.barrios
        umbral = 8   # umbral de equilibrio entre barrios

        # Recogemos inventarios de cada barrio de forma segura
        almacen_a: dict[str, int] = {}
        almacen_b: dict[str, int] = {}

        for calle in barrio_a.calles:
            with calle.cerrojo:
                for item, cant in calle.inventario.items():
                    almacen_a[item] = almacen_a.get(item, 0) + cant

        for calle in barrio_b.calles:
            with calle.cerrojo:
                for item, cant in calle.inventario.items():
                    almacen_b[item] = almacen_b.get(item, 0) + cant

        # Si barrio_a tiene exceso y barrio_b déficit, transferimos
        for item in almacen_a:
            exceso = almacen_a[item] - umbral
            deficit = umbral - almacen_b.get(item, 0)
            if exceso > 0 and deficit > 0:
                mover = min(exceso, deficit)
                # Retiramos de la calle con más stock en barrio_a
                for calle in sorted(barrio_a.calles,
                                    key=lambda c: c.inventario.get(item, 0),
                                    reverse=True):
                    with calle.cerrojo:
                        disponible = calle.inventario.get(item, 0)
                        a_retirar = min(disponible, mover)
                        if a_retirar > 0:
                            calle.inventario[item] -= a_retirar
                            mover -= a_retirar
                    if mover <= 0:
                        break
                # Depositamos en la calle con menos stock en barrio_b
                ya_movido = min(exceso, deficit) - mover
                if ya_movido > 0:
                    calle_destino = min(barrio_b.calles,
                                       key=lambda c: c.inventario.get(item, 0))
                    with calle_destino.cerrojo:
                        calle_destino.inventario[item] =                             calle_destino.inventario.get(item, 0) + ya_movido
                    print(f"[Pueblo {self.nombre}] {ya_movido} {item}: "
                          f"{barrio_a.nombre} → {barrio_b.nombre}.")

# Definimos los métodos públicos
    # Método para añadir un barrio al pueblo
    def añadir_barrio(self, barrio: Barrio) -> None:
        if len(self.barrios) >= 2:
            raise ValueError(f"{self.nombre}: máximo 2 barrios por pueblo.")
        self.barrios.append(barrio)
        print(f"{barrio.nombre} se une al pueblo {self.nombre}.")

    # Método para iniciar el pueblo en cascada
    def iniciar(self) -> None:
        for barrio in self.barrios:
            barrio.iniciar()
        self._hilo_pueblo = Thread(target=self._logica_pueblo, daemon=True)
        self._hilo_pueblo.start()
        print(f"\n── {self.nombre} fundado ──")

    # Detiene el pueblo, los barrios, las calles y los trabajadores
    def detener(self) -> None:
        self._activo.clear()
        for barrio in self.barrios:
            barrio.detener()
        if self._hilo_pueblo:
            self._hilo_pueblo.join(timeout=5)
        print(f"\n── {self.nombre} disuelto ──")

    # Método para consultar las reservas totales del pueblo
    def consultar_reservas(self) -> None:
        print(f"\n── Reservas del pueblo {self.nombre} ──")
        total_pueblo: dict[str, int] = {}
        for barrio in self.barrios:
            print(f"  {barrio.nombre}:")
            for calle in barrio.calles:
                with calle.cerrojo:
                    for item, cantidad in calle.inventario.items():
                        total_pueblo[item] = total_pueblo.get(item, 0) + cantidad
                        if cantidad > 0:
                            print(f"    {calle.nombre} → {item}: {cantidad}")
        print(f"  ────────────────────────")
        for item, total in total_pueblo.items():
            print(f"  {item}: {total} (total pueblo)")

    # Riqueza total del pueblo sumando todos los barrios
    def total_monedas(self) -> None:
        total_pueblo = 0
        print(f"\n── Riqueza total de {self.nombre} ──")
        for barrio in self.barrios:
            total_barrio = 0
            for calle in barrio.calles:
                for jugador in calle.jugadores:
                    total_barrio += jugador.monedas
            total_pueblo += total_barrio
            print(f"  {barrio.nombre}: {total_barrio} monedas")
        print(f"  TOTAL PUEBLO: {total_pueblo} monedas")

    # Método para cambiar de jugador activo en cualquier barrio del pueblo
    def cambiar_jugador(self, nombre: str) -> "tuple[Jugador, Calle, Barrio] | None":
        nombre = nombre.lower().strip("[]")
        for barrio in self.barrios:
            for calle in barrio.calles:
                for jugador in calle.jugadores:
                    if jugador.nombre.lower() == nombre:
                        return jugador, calle, barrio
        print(f"No existe ningún jugador llamado '{nombre}' en {self.nombre}.")
        return None

    # Ranking de todos los jugadores del pueblo por monedas
    def ranking_jugadores(self) -> None:
        jugadores = []
        for barrio in self.barrios:
            for calle in barrio.calles:
                for jugador in calle.jugadores:
                    jugadores.append((jugador.nombre, jugador.monedas,
                                      calle.nombre, barrio.nombre))
        jugadores.sort(key=lambda x: x[1], reverse=True)
        print(f"\n── Ranking del pueblo {self.nombre} ──")
        for pos, (nombre, monedas, calle, barrio) in enumerate(jugadores, 1):
            print(f"  {pos}. {nombre} ({calle}, {barrio}): {monedas} monedas")


#### MUD para semana 3

In [ ]:
# Extendemos mud() con el parámetro pueblo y los nuevos comandos.
_mud_semana2 = mud

def mud(jugador_activo: Jugador, calle: Calle,
        barrio: Barrio | None = None,
        pueblo: "Pueblo | None" = None) -> None:
    # Mensaje de ayuda del juego con los distintos comandos
    ayuda = """
COMANDOS:


NO ES NECESARIO PONER COMILLAS NI SÍMBOLOS DE APERTURA Y CIERRE


producir [item]                        → empezar a producir
parar                                  → parar producción
inventario                             → ver inventario de la calle
bolsa                                  → ver tu bolsa y monedas
recetas                                → ver tus recetas
precios                                → ver tus precios de producción
precio [item] [num]                    → fijar precio de producción
precio_bolsa [item] [num]              → fijar precio de reventa personal
comprar [item] [num]                   → comprar item de la calle
vender [item] [num] [jugador]          → vender desde tu bolsa (pide confirmación)
depositar [item] [num]                 → depositar item de la bolsa al taller
retirar [item] [num]                   → retirar item del taller a la bolsa
eliminar [item] [num]                  → eliminar stock de la calle
almacen                                → ver almacén del barrio
reservas                               → ver reservas totales del pueblo
cambiar [jugador]                      → cambiar de jugador activo
ranking                                → ranking de calles por riqueza
riqueza                                → riqueza total
ayuda                                  → mostrar este menú
salir                                  → cerrar el juego
"""
    # Mensaje de inicio del juego
    print(f"\nBienvenido a {calle.nombre}, {jugador_activo.nombre}.")
    print(ayuda)
    # Se genera el bucle MUD
    while True:
        try:
            entrada = input(f"[{jugador_activo.nombre}] > ").strip().lower()
        except KeyboardInterrupt:
            print("\nInterrupción, cerrando...")
            if pueblo:
                pueblo.detener()
            elif barrio:
                barrio.detener()
            else:
                calle.detener()
            break
        # En el caso de que no haya inputs se prosigue
        if not entrada:
            continue
        partes = entrada.split()
        comando = partes[0]
        # Parte de la producción del MUD, para empezar a producir
        if comando == "producir":
            if len(partes) < 2:
                jugador_activo.trabajador.consultar_recetas()
                print("Uso: producir [item]")
            else:
                item = partes[1].strip("[]")
                jugador_activo.trabajador.asignar_tarea(item)
                jugador_activo.estado = EstadoJugador.PRODUCIENDO
        # Parte de la producción del MUD, para pararla concretamente
        elif comando == "parar":
            jugador_activo.trabajador.parar()
            jugador_activo.estado = EstadoJugador.DESCANSANDO
        # Parte de las consultas del MUD, para consultar el inventario
        elif comando == "inventario":
            calle.consultar_inventario()
        # Parte de las consultas del MUD, para consultar la bolsa
        elif comando == "bolsa":
            jugador_activo.consultar_bolsa()
        # Parte de las consultas del MUD, para consultar las recetas
        elif comando == "recetas":
            jugador_activo.trabajador.consultar_recetas()
        # Parte de las consultas del MUD, para consultar los precios
        elif comando == "precios":
            precios = jugador_activo.trabajador.consultar_precios()
            print(f"\n── Precios de {jugador_activo.trabajador.nombre} ──")
            for item, precio in precios.items():
                print(f"  {item}: {precio} monedas")
        # Parte de las peticiones del MUD, para fijar precios
        elif comando == "precio":
            if len(partes) < 3:
                print("Uso: precio [item] [cantidad]")
            else:
                try:
                    jugador_activo.trabajador.fijar_precio(partes[1].strip("[]"), int(partes[2]))
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para comprar elementos
        elif comando == "comprar":
            if len(partes) < 3:
                print("Uso: comprar [item] [cantidad]")
            else:
                try:
                    item_c = partes[1].strip("[]")
                    cant_c = int(partes[2])

                    calle_origen = None

                    # 1. Busca stock en la calle actual
                    if calle.inventario.get(item_c, 0) >= cant_c:
                        calle_origen = calle

                    # 2. Si no, busca en el resto del barrio
                    if calle_origen is None and barrio:
                        for c in barrio.calles:
                            if c is calle:
                                continue
                            if c.inventario.get(item_c, 0) >= cant_c:
                                calle_origen = c
                                break

                    # 3. Si no, busca en el resto del pueblo
                    if calle_origen is None and pueblo:
                        for b in pueblo.barrios:
                            for c in b.calles:
                                if c is calle:
                                    continue
                                if c.inventario.get(item_c, 0) >= cant_c:
                                    calle_origen = c
                                    break
                            if calle_origen:
                                break

                    if calle_origen is None:
                        print(f"No hay suficiente '{item_c}' disponible en el pueblo.")
                    else:
                        # Obtiene el precio según el tipo de barrio
                        barrio_origen = barrio
                        if pueblo:
                            for b in pueblo.barrios:
                                if calle_origen in b.calles:
                                    barrio_origen = b
                                    break
                        if barrio_origen:
                            precio_u = barrio_origen.obtener_precio(item_c, calle_origen)
                        else:
                            precio_u = None
                            for j in calle_origen.jugadores:
                                if item_c in j.trabajador.precios:
                                    precio_u = j.trabajador.precios[item_c]
                                    break

                        if precio_u is None:
                            print(f"No hay precio fijado para '{item_c}'.")
                        else:
                            if calle_origen is not calle:
                                print(f"(Comprando de {calle_origen.nombre})")
                            jugador_activo.comprar(item_c, cant_c,
                                                   calle_origen.inventario,
                                                   calle_origen.cerrojo,
                                                   precio_u)
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para vender elementos
        elif comando == "vender":
            if len(partes) < 4:
                print("Uso: vender [item] [cantidad] [comprador]")
            else:
                try:
                    item_v   = partes[1].strip("[]")
                    cant_v   = int(partes[2])
                    nombre_c = partes[3].strip("[]")

                    comprador_obj = None
                    if pueblo:
                        res = pueblo.cambiar_jugador(nombre_c)
                        if res:
                            comprador_obj = res[0]
                    elif barrio:
                        res = barrio.cambiar_jugador(nombre_c)
                        if res:
                            comprador_obj = res[0]
                    else:
                        comprador_obj = calle.cambiar_jugador(nombre_c)

                    if comprador_obj is None:
                        print(f"No se encontró al jugador '{nombre_c}'.")
                    elif comprador_obj is jugador_activo:
                        print("No puedes venderte a ti mismo.")
                    else:
                        # Precio de bolsa tiene prioridad sobre precio del trabajador
                        precio_u = jugador_activo.precios_bolsa.get(item_v) \
                                   or jugador_activo.trabajador.precios.get(item_v)
                        if precio_u is None:
                            print(f"Sin precio para '{item_v}'. Usa: precio_bolsa {item_v} [num]")
                        else:
                            coste = precio_u * cant_v
                            print(f"\n{jugador_activo.nombre} ofrece {cant_v} x {item_v} "
                                  f"a {comprador_obj.nombre} por {coste} monedas.")
                            # ── Confirmación del comprador ──
                            try:
                                resp = input(f"[{comprador_obj.nombre}] ¿Aceptas? (s/n) > ").strip().lower()
                            except KeyboardInterrupt:
                                resp = "n"
                            if resp == "s":
                                jugador_activo.vender(item_v, cant_v, precio_u, comprador_obj)
                            else:
                                print(f"{comprador_obj.nombre} rechazó la oferta.")
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para depositar items de la bolsa al taller
        elif comando == "depositar":
            if len(partes) < 3:
                print("Uso: depositar [item] [cantidad]")
            else:
                try:
                    item_d = partes[1].strip("[]")
                    cant_d = int(partes[2])
                    with jugador_activo.cerrojo:
                        if jugador_activo.inventario.get(item_d, 0) < cant_d:
                            print(f"No tienes suficiente {item_d} en la bolsa.")
                        else:
                            with calle.cerrojo:
                                jugador_activo.inventario[item_d] -= cant_d
                                calle.inventario[item_d] = calle.inventario.get(item_d, 0) + cant_d
                            print(f"Has depositado {cant_d} {item_d} en el taller.")
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para retirar items del taller a la bolsa
        elif comando == "retirar":
            if len(partes) < 3:
                print("Uso: retirar [item] [cantidad]")
            else:
                try:
                    item_r = partes[1].strip("[]")
                    cant_r = int(partes[2])
                    with calle.cerrojo:
                        if calle.inventario.get(item_r, 0) < cant_r:
                            print(f"No hay suficiente {item_r} en el taller.")
                        else:
                            with jugador_activo.cerrojo:
                                calle.inventario[item_r] -= cant_r
                                jugador_activo._añadir_a_inventario(item_r, cant_r)
                            print(f"Has retirado {cant_r} {item_r} a tu bolsa.")
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para eliminar stock
        elif comando == "eliminar":
            if len(partes) < 3:
                print("Uso: eliminar [item] [cantidad]")
            else:
                try:
                    jugador_activo.trabajador.eliminar_stock(partes[1].strip("[]"), int(partes[2]))
                except ValueError:
                    print("La cantidad tiene que ser un número entero.")
        # Parte de las peticiones del MUD, para recibir la ayuda
        elif comando == "ayuda":
            print(ayuda)
        # Parte de las peticiones del MUD, para salir del juego
        elif comando == "salir":
            print(f"\nHasta luego, {jugador_activo.nombre}.")
            if pueblo:
                pueblo.detener()
            elif barrio:
                barrio.detener()
            else:
                calle.detener()
            break
        # Parte de las peticiones del MUD, para cambiar de jugador activo
        elif comando == "cambiar":
            if len(partes) < 2:
                if pueblo:
                    print("Jugadores disponibles en el pueblo:")
                    for b in pueblo.barrios:
                        print(f"  [{b.nombre}]")
                        for c in b.calles:
                            marca = " ◄ (aquí)" if c.nombre == calle.nombre else ""
                            print(f"    [{c.nombre}]{marca}")
                            for j in c.jugadores:
                                print(f"      {j.nombre} | {j.estado.name} | {j.monedas} monedas")
                elif barrio:
                    print("Jugadores disponibles en el barrio:")
                    for c in barrio.calles:
                        marca = " ◄ (aquí)" if c.nombre == calle.nombre else ""
                        print(f"  [{c.nombre}]{marca}")
                        for j in c.jugadores:
                            print(f"    {j.nombre} | {j.estado.name} | {j.monedas} monedas")
                else:
                    print("Jugadores disponibles:")
                    for j in calle.jugadores:
                        print(f"  {j.nombre} | {j.estado.name} | {j.monedas} monedas")
                print("Uso: cambiar [nombre]")
            else:
                if pueblo:
                    resultado = pueblo.cambiar_jugador(partes[1])
                    if resultado is not None:
                        jugador_activo, calle, barrio = resultado   # ← actualiza LAS TRES
                        print(f"Ahora controlas a {jugador_activo.nombre} "
                              f"en {calle.nombre} ({barrio.nombre}).")
                elif barrio:
                    resultado = barrio.cambiar_jugador(partes[1])
                    if resultado is not None:
                        jugador_activo, calle = resultado
                        print(f"Ahora controlas a {jugador_activo.nombre} "
                              f"en {calle.nombre}.")
                else:
                    resultado = calle.cambiar_jugador(partes[1])
                    if resultado is not None:
                        jugador_activo = resultado
                        print(f"Ahora controlas a {jugador_activo.nombre}.")
        # Parte de las consultas del MUD, para ver el ranking
        elif comando == "ranking":
            if pueblo:
                pueblo.ranking_jugadores()
            elif barrio:
                barrio.ranking_calles()
            else:
                print("No estás en un barrio todavía.")
        # Parte de las consultas del MUD, para ver el precio de la bolsa
        elif comando == "precio_bolsa":
            if len(partes) < 3:
                print("Uso: precio_bolsa [item] [num]")
            else:
                try:
                    jugador_activo.fijar_precio_bolsa(partes[1].strip("[]"), int(partes[2]))
                except ValueError:
                    print("El precio tiene que ser un número entero.")
        # Parte de las consultas del MUD, para ver la riqueza total
        elif comando == "riqueza":
            if pueblo:
                pueblo.total_monedas()
            elif barrio:
                barrio.total_monedas()
            else:
                jugador_activo.consultar_bolsa()
        # Parte de las consultas del MUD, para ver todos los objetos del barrio
        elif comando == "almacen":
            if barrio:
                barrio.consultar_almacen()
            else:
                print("No estás en un barrio todavía.")
        # Parte de las consultas del MUD, para ver las reservas del pueblo
        elif comando == "reservas":
            if pueblo:
                pueblo.consultar_reservas()
            else:
                print("No estás en un pueblo todavía.")
        # Control de errores, para cuando el comando sea desconocido
        else:
            print(f"Comando '{comando}' desconocido. Escribe 'ayuda' para ver los comandos.")


#### EJEMPLO DE FORMA LOCAL

In [ ]:

# EJEMPLO PARA LA SEMANA 3 — MODO LOCAL (sin PyRO4)
if __name__ == "__main__":


    # ── Calle del Ébano (madera/carpintería) ──
    recetas_lenador = [
        {"produce": "madera", "cantidad": 1, "necesita": {}, "tiempo": 3}
    ]
    recetas_carpintero = [
        {"produce": "tablas",  "cantidad": 1, "necesita": {"madera": 2}, "tiempo": 4},
        {"produce": "muebles", "cantidad": 1, "necesita": {"madera": 5}, "tiempo": 8},
    ]
    t_lenador    = Trabajador(recetas_lenador,    {}, Lock(), {"madera": 3})
    t_carpintero = Trabajador(recetas_carpintero, {}, Lock(), {"tablas": 5, "muebles": 20})
    marc  = Jugador("Marc",  t_lenador,    monedas=50)
    pedro = Jugador("Pedro", t_carpintero, monedas=50)
    calle_ebano = Calle("Calle del Ébano")
    calle_ebano.añadir_jugador(marc)
    calle_ebano.añadir_jugador(pedro)


    # ── Calle del Hierro (minería/herrería) ──
    recetas_minero = [
        {"produce": "mineral", "cantidad": 1, "necesita": {}, "tiempo": 4}
    ]
    recetas_herrero = [
        {"produce": "hierro",       "cantidad": 1, "necesita": {"mineral": 2}, "tiempo": 5},
        {"produce": "herramientas", "cantidad": 1, "necesita": {"hierro": 1},  "tiempo": 6},
    ]
    t_minero  = Trabajador(recetas_minero,  {}, Lock(), {"mineral": 2})
    t_herrero = Trabajador(recetas_herrero, {}, Lock(), {"hierro": 8, "herramientas": 15})
    ana    = Jugador("Ana",    t_minero,  monedas=50)
    carlos = Jugador("Carlos", t_herrero, monedas=50)
    calle_hierro = Calle("Calle del Hierro")
    calle_hierro.añadir_jugador(ana)
    calle_hierro.añadir_jugador(carlos)


    # ── Calle de la Harina (agricultura/molinería) ──
    recetas_agricultor = [
        {"produce": "trigo", "cantidad": 1, "necesita": {}, "tiempo": 3}
    ]
    recetas_molinero = [
        {"produce": "harina", "cantidad": 1, "necesita": {"trigo": 2}, "tiempo": 4},
        {"produce": "pan",    "cantidad": 1, "necesita": {"harina": 1}, "tiempo": 3},
    ]
    t_agricultor = Trabajador(recetas_agricultor, {}, Lock(), {"trigo": 2})
    t_molinero   = Trabajador(recetas_molinero,   {}, Lock(), {"harina": 4, "pan": 6})
    lucia = Jugador("Lucia", t_agricultor, monedas=50)
    diego = Jugador("Diego", t_molinero,   monedas=50)
    calle_harina = Calle("Calle de la Harina")
    calle_harina.añadir_jugador(lucia)
    calle_harina.añadir_jugador(diego)


    # ── Calle de las Telas (ganadería/tejidos) ──
    recetas_ganadero = [
        {"produce": "lana", "cantidad": 1, "necesita": {}, "tiempo": 5}
    ]
    recetas_tejedor = [
        {"produce": "tela",  "cantidad": 1, "necesita": {"lana": 2}, "tiempo": 6},
        {"produce": "ropa",  "cantidad": 1, "necesita": {"tela": 2}, "tiempo": 8},
    ]
    t_ganadero = Trabajador(recetas_ganadero, {}, Lock(), {"lana": 3})
    t_tejedor  = Trabajador(recetas_tejedor,  {}, Lock(), {"tela": 5, "ropa": 12})
    elena = Jugador("Elena", t_ganadero, monedas=50)
    felix = Jugador("Felix", t_tejedor,  monedas=50)
    calle_telas = Calle("Calle de las Telas")
    calle_telas.añadir_jugador(elena)
    calle_telas.añadir_jugador(felix)


    # ── Barrio Comercial (Ébano + Hierro) ──
    barrio_comercial = BarrioComercial("Barrio del Mercado", intervalo=15)
    barrio_comercial.añadir_calle(calle_ebano)
    barrio_comercial.añadir_calle(calle_hierro)


    # ── Barrio Económico (Harina + Telas) ──
    barrio_economico = BarrioEconomico("Barrio de la Economía", intervalo=20)
    barrio_economico.añadir_calle(calle_harina)
    barrio_economico.añadir_calle(calle_telas)


    # ── Pueblo ──
    pueblo = Pueblo("Villa Medieva")
    pueblo.añadir_barrio(barrio_comercial)
    pueblo.añadir_barrio(barrio_economico)
    pueblo.iniciar()


    # ── MUD: Marc arranca el juego ──
    mud(marc, calle_ebano, barrio_comercial, pueblo)

#### EJEMPLO MODO DISTRIBUIDO (Con PyRO4)

In [ ]:

# EJEMPLO PARA LA SEMANA 3 — MODO DISTRIBUIDO (con PyRO4)
#
# PASOS PREVIOS (en terminales separadas):
#   1.  pyro4-ns                          ← nameserver
#   2.  python script_calle_ebano.py      ← nodo 1
#   3.  python script_calle_hierro.py     ← nodo 2
#   4.  python script_barrio.py           ← nodo barrio (este archivo)
#
# script_calle_ebano.py (ejemplo mínimo):
#   calle = Calle("Calle del Ébano")
#   ... (añadir jugadores)
#   calle.iniciar(distribuido=True)       ← registra en nameserver
#   mud(marc, calle)
#
# script_barrio.py (ejemplo mínimo):
if __name__ == "__main__":
    barrio = BarrioComercial("Barrio del Mercado", distribuido=True, intervalo=15)
    barrio.añadir_calle("Calle del Ébano")   # nombre str, no objeto
    barrio.añadir_calle("Calle del Hierro")
    barrio.iniciar()
    input("Barrio corriendo. Pulsa Enter para detener.\n")
    barrio.detener()